In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
from scipy.interpolate import RegularGridInterpolator


import os

In [ ]:
os.chdir('../..')

In [ ]:
### read in surface mass balance from “Analysis of the Surface Mass Balance for Deglacial Climate Simulations”
#submitted to The Cryosphere by M.-L. Kapsch 1, U. Mikolajewicz 1, F. Ziemen 1*, 
#Christian B. Rodehacke 2,3, and C. Schannwell 1.

smb = nc.Dataset('inputs/Glac1D_smb/Src/smb_2D_21000.nc')

In [ ]:
smb

In [ ]:
smb['XLONGLOB1'][:].shape, smb['YLATGLOBP5'][:].shape

In [ ]:
smb['smb']

In [ ]:
fig,ax = plt.subplots()
year = 365.25 * 24*60*60
val = 2
img = ax.pcolormesh(smb['XLONGLOB1'][:], smb['YLATGLOBP5'][:], np.squeeze(smb['smb'][:,:,:])*year,vmin = -val,vmax = val,cmap = 'bwr_r')
ax.contour(smb['XLONGLOB1'][:], smb['YLATGLOBP5'][:], np.squeeze(smb['smb'][:,:,:])*year,levels = [0])
plt.colorbar(img, ax = ax)

In [ ]:
f = RegularGridInterpolator((smb['XLONGLOB1'][:], smb['YLATGLOBP5'][:]), np.squeeze(smb['smb'][:,:,:]).T*year,method = 'cubic')

In [ ]:
### read in transects
jdf = pd.read_excel('inputs/flowlines/jdf_update.xlsx')
sv = pd.read_excel('inputs/flowlines/sv.xlsx')
ysv = pd.read_excel('inputs/flowlines/ysv_update.xlsx')

In [ ]:
flowlines = [jdf, sv, ysv]

In [ ]:
fig,ax = plt.subplots()
year = 365.25 * 24*60*60
val = 2
img = ax.pcolormesh(smb['XLONGLOB1'][:], smb['YLATGLOBP5'][:], np.squeeze(smb['smb'][:,:,:])*year,vmin = -val,vmax = val,cmap = 'bwr_r')
plt.colorbar(img, ax = ax)

for flow in flowlines:
    ax.plot(flow['Lon']+360, flow['Lat'],color = 'k')

ax.set_xlim(150,250)
ax.set_ylim(30,80)

In [ ]:
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

In [ ]:
fig,ax = plt.subplots(figsize=(4,4))
lbs = ['Juan de Fuca', 'Skeena Valley', 'Yakutat Sea Valley']
for i, flow in enumerate(flowlines):
    flow['smb'] = f((flow['Lon']+360, flow['Lat']))
    flow['smb_up_ave'] = np.cumsum(flow['smb']) / (np.arange(len(flow['distance']))+1)
    plt.hist(flow['smb_up_ave'],bins = np.linspace(-3.5,1.5,30),alpha = 0.5,label = lbs[i])
    # ax.axvline(np.mean(flow['smb_up_ave']),color = colors[i])

ax.set_xlabel('Upstream Average SMB (m/yr)')
ax.set_ylabel('Count')
ax.legend(fontsize=10)
plt.tight_layout()

# plt.savefig('figures/sf_21kaSMB.pdf')